# NCAA FBS Elo Rating Model

Prospective NCAA FBS Elo ratings with margin-of-victory adjustment, home-field advantage, offseason regression, and expanding-window parameter tuning.


In [1]:
# pip install numpy pandas pyarrow scikit-learn scipy ipython tzdata


## 1. Setup and NCAA FBS Data

Load public [SportsDataverse college schedules](https://py.sportsdataverse.org/docs/0.1.5/cfb/reference/loaders/schedule)
and the [ESPN schedule master](https://github.com/sportsdataverse/cfbfastR-cfb-raw/blob/main/cfb/cfb_schedule_master.parquet).
The loading cell maps these into the original NFL column names. No API key is needed.
Run from this notebook's directory and keep the accompanying `data/cfb_overtime_supplement.csv`.
Files are cached in `data/`; the current season and overtime metadata refresh by default.
Set `refresh_data = False` to replay the included snapshot offline.

Retain regular-season and postseason games involving at least one season-specific FBS team,
including conference championships, bowls, and CFP games. January games retain their source
football season. FCS opponents have individual ratings, start at the same 1000 as every new
team, and carry over normally; FCS-only games are excluded. This is the closest transfer of the
original equal-start methodology, not a separately calibrated FCS-strength model. Rankings are
FBS-only; projections retain FCS opponents. Team IDs are rating keys, with school names alongside.

The original 0.75/0.25 overtime result weights are retained, including multi-overtime games.
Overtime is read from ESPN final-period/status metadata. A 17-row, source-linked supplement
covers historical gaps; missing completed-game overtime metadata raises an error rather than
silently treating a game as regulation. Duplicate source fixtures are counted only once.

All original parameter grids and training/evaluation years are retained. They are inherited
search choices, not established college-football parameter values. Full tuning is computationally
expensive because the original row-by-row calculations and repeated fitting are preserved.


In [2]:
import numpy as np 
import pandas as pd
from pathlib import Path
from urllib.request import urlretrieve
from sklearn.metrics import accuracy_score, log_loss

#Load college games and map them to the original schedule columns.
refresh_data = True
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
schedule_season = pd.Timestamp.now(tz="America/Chicago").year
games = []
for season in range(2003, schedule_season + 1):
    schedule_file = data_dir / f"cfb_schedules_{season}.parquet"
    if not schedule_file.exists() or (refresh_data and season == schedule_season):
        urlretrieve(
            f"https://github.com/sportsdataverse/sportsdataverse-data/releases/download/cfb_schedules/cfb_schedules_{season}.parquet",
            schedule_file
        )
    games.append(pd.read_parquet(schedule_file))
games = pd.concat(games, ignore_index=True)
games = games[games["fbs_participant"].fillna(False) &
              games["season_type"].isin(["regular", "postseason"]) &
              ~games["status"].isin(["STATUS_CANCELED", "STATUS_CANCELLED"])].copy()

# Some historical source fixtures have both a placeholder and an ESPN game ID.
games["fixture_date"] = games["start_date"].str[:10]
games = (games.sort_values("game_id")
         .drop_duplicates(["season", "home_team", "away_team", "fixture_date"], keep="last")
         .drop(columns="fixture_date"))

overtime_file = data_dir / "cfb_schedule_master.parquet"
if not overtime_file.exists() or refresh_data:
    urlretrieve(
        "https://raw.githubusercontent.com/sportsdataverse/cfbfastR-cfb-raw/main/cfb/cfb_schedule_master.parquet",
        overtime_file
    )
overtime_games = pd.read_parquet(overtime_file).set_index("game_id")
overtime = ((overtime_games["status_period"] > 4) |
            overtime_games["status_type_detail"].str.contains("OT", na=False)).astype(float)
overtime = overtime.where(overtime_games["status_type_completed"])
overtime_supplement = pd.read_csv(data_dir / "cfb_overtime_supplement.csv").set_index("game_id")
games["overtime"] = games["game_id"].map(overtime).fillna(
    games["game_id"].map(overtime_supplement["overtime"]))
games["start_date"] = games["game_id"].map(overtime_supplement["start_date"]).fillna(games["start_date"])
if games.loc[games["completed"], "overtime"].isna().any():
    raise ValueError("Missing completed-game overtime metadata; refresh the schedule master before fitting.")

games = games.rename(columns={"home_team": "home_team_name", "away_team": "away_team_name",
                              "home_points": "home_score", "away_points": "away_score"})
games["home_team"] = games["home_id"].astype("int64").astype(str)
games["away_team"] = games["away_id"].astype("int64").astype(str)
games["game_type"] = games["season_type"].map({"regular": "REG", "postseason": "POST"})
games["location"] = np.where(games["neutral_site"], "Neutral", "Home")
games["kickoff"] = pd.to_datetime(games["start_date"], utc=True, format="mixed").dt.tz_convert("America/Chicago")
games["gameday"] = games["kickoff"].dt.strftime("%Y-%m-%d")
games["gametime"] = games["kickoff"].dt.strftime("%H:%M:%S")
games.loc[games["start_time_tbd"].fillna(False), "gametime"] = ""
# A live or scheduled score (including a placeholder 0-0) is not a final result.
games.loc[~games["completed"], ["home_score", "away_score"]] = np.nan
games = games.sort_values(["season", "gameday", "gametime", "game_id"]).reset_index(drop=True)
assert games["game_id"].is_unique
assert games.loc[games["completed"], ["home_score", "away_score"]].notna().all().all()
print(games.columns)
print(games.shape)
print(games.head())


Index(['game_id', 'season', 'week', 'season_type', 'season_type_id',
       'start_date', 'start_time_tbd', 'completed', 'neutral_site',
       'conference_game', 'conference_competition', 'attendance', 'venue_id',
       'venue', 'status', 'home_id', 'home_team_name', 'home_abbreviation',
       'home_division', 'home_conference', 'home_score', 'home_winner',
       'away_id', 'away_team_name', 'away_abbreviation', 'away_division',
       'away_conference', 'away_score', 'away_winner', 'fbs_game',
       'fbs_participant', 'highlights', 'notes', 'playoff_competition',
       'playoff_format', 'playoff_round', 'playoff_round_name',
       'playoff_bracket_slot', 'playoff_home_seed', 'playoff_away_seed',
       'playoff_bowl_name', 'home_rank', 'away_rank', 'overtime', 'home_team',
       'away_team', 'game_type', 'location', 'kickoff', 'gameday', 'gametime'],
      dtype='str')
(19971, 51)
     game_id  season  week season_type  season_type_id  \
0  232350023    2003     1     regular 

In [3]:
games['location'].unique()

<ArrowStringArray>
['Home', 'Neutral']
Length: 2, dtype: str

In [4]:
games['season'].unique()

array([2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013,
       2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024,
       2025, 2026])

## 2. Core Elo Rating Functions


In [5]:
import numpy as np
import pandas as pd
from IPython.display import display


def null_coalesce(x, y):
    return y if x is None else x

def compress_exp_grid(min_val, max_val, n=10, curve=2):
    grid = np.unique(np.round(
        min_val + (max_val - min_val) *
        (np.exp(curve * np.linspace(0, 1, n)) - 1) / (np.exp(curve) - 1),2))
    return grid

def calc_mov_scale(games):
    mean_log_margin = np.log(
        (games["home_score"] - games["away_score"]).abs() + 1
    ).mean()
    return mean_log_margin

def cfb_home_result(home_score, away_score, overtime):
    conditions = [
        (home_score > away_score) & (overtime == 0),
        (home_score < away_score) & (overtime == 0),
        (home_score > away_score) & (overtime == 1),
        (home_score < away_score) & (overtime == 1),
        home_score == away_score
    ]
    choices = [1.00, 0.00, 0.75, 0.25, 0.50]
    return np.select(conditions, choices, default=np.nan)

def elo_season(games, K, ratings=None, scale=400, start_rating=1000,
               use_mov=True, mov_scale=1, use_home_adv=True, home_adv=0):

    ratings = {} if ratings is None else ratings.copy()
    n = len(games)

    if use_mov and (pd.isna(mov_scale) or mov_scale <= 0):
        mov_scale = 1
    if not use_home_adv:
        home_adv = 0

    elo_home_pre = np.zeros(n)
    elo_away_pre = np.zeros(n)
    elo_home_post = np.zeros(n)
    elo_away_post = np.zeros(n)
    p_home = np.zeros(n)

    for i in range(n):
        g = games.iloc[i]

        hid = str(g["home_team"])
        aid = str(g["away_team"])

        Ra = ratings.get(hid, start_rating)
        Rb = ratings.get(aid, start_rating)

        elo_home_pre[i] = Ra
        elo_away_pre[i] = Rb

        game_home_adv = 0 if g["location"] == "Neutral" else home_adv

        p = 1 / (1 + 10 ** ((Rb - Ra - game_home_adv) / scale))
        p_home[i] = p

        y = cfb_home_result(home_score=g["home_score"], away_score=g["away_score"], overtime=g["overtime"])

        if pd.isna(y):
            raise ValueError(f"Invalid NCAA FBS result coding for game_id: {g['game_id']}")

        point_diff = abs(g["home_score"] - g["away_score"])
        if g["home_score"] > g["away_score"]:
            winner_elo_diff = (Ra + game_home_adv) - Rb
        elif g["home_score"] < g["away_score"]:
            winner_elo_diff = Rb - (Ra + game_home_adv)
        else:
            winner_elo_diff = 0

        if not use_mov:
          mov_mult = 1
        elif g["home_score"] == g["away_score"]:
          mov_mult = 1
        else:
          mov_mult = np.log(point_diff + 1) * (2.2 / (0.001 * winner_elo_diff + 2.2)) / mov_scale

        delta = K * (y - p) * mov_mult

        ratings[hid] = Ra + delta
        ratings[aid] = Rb - delta

        elo_home_post[i] = ratings[hid]
        elo_away_post[i] = ratings[aid]

    games_out = games.copy()
    games_out["elo_home_pre"] = elo_home_pre
    games_out["elo_away_pre"] = elo_away_pre
    games_out["elo_home_post"] = elo_home_post
    games_out["elo_away_post"] = elo_away_post
    games_out["p_home"] = p_home
    games_out["p_away"] = 1 - p_home

    return {"games": games_out, "ratings": ratings}

### Single-Season Elo Sanity Check


In [6]:
# Sort it chronologically, calculate its MOV scale, and run
test_games = games[(games["season"] == 2024) & games["home_score"].notna() & games["away_score"].notna()].copy()

test_games = test_games.sort_values(["gameday", "game_id"]).reset_index(drop=True)
mov_scale = calc_mov_scale(test_games)
test_elo = elo_season(games=test_games, K=20, ratings={}, scale=400, start_rating=1000, use_mov=True, mov_scale=mov_scale, use_home_adv=True, home_adv=50)

test_elo["games"][["gameday", "home_team", "away_team", "elo_home_pre", "elo_away_pre", "p_home", "elo_home_post", "elo_away_post"]].head(20)
display(test_elo["ratings"])

#Sanity Check - average of elo values should still be around starting mean of 1000
print("Average ELO Rank: ",
      sum(test_elo["ratings"].values())
      / len(test_elo["ratings"]))

{'59': np.float64(1012.000035925813),
 '52': np.float64(934.1895609874301),
 '2440': np.float64(949.7904290602789),
 '2567': np.float64(1075.5921617725185),
 '167': np.float64(980.2561353784771),
 '147': np.float64(1007.1856321733808),
 '62': np.float64(981.0036593064578),
 '2169': np.float64(990.1110161777267),
 '8': np.float64(1020.3478433887287),
 '2029': np.float64(983.7268716064854),
 '142': np.float64(1050.4532235208496),
 '93': np.float64(974.7369355083204),
 '135': np.float64(1045.5589463936567),
 '153': np.float64(995.3584596389651),
 '164': np.float64(1011.5396716915409),
 '47': np.float64(988.3624921920509),
 '154': np.float64(963.2985930206611),
 '2448': np.float64(988.8138363573003),
 '2116': np.float64(970.4207703136897),
 '160': np.float64(987.179583057435),
 '38': np.float64(1047.7656861347373),
 '2449': np.float64(994.2677330797089),
 '152': np.float64(987.7035150911543),
 '2717': np.float64(990.7530097093502),
 '2433': np.float64(967.9380918420787),
 '2296': np.float6

Average ELO Rank:  999.9999999999989


## 3. Offseason Regression and Evaluation


In [7]:
def carry_over(ratings, carry=0.6):
    if len(ratings) == 0:
        return ratings

    mu = np.mean(list(ratings.values()))
    return {team: carry * rating + (1 - carry) * mu
            for team, rating in ratings.items()}

def logloss(p, y, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -(y * np.log(p) + (1 - y) * np.log1p(-p))

### Carryover Sanity Check


In [8]:
#Sanity Checks
test_carry = carry_over(test_elo["ratings"], carry=0.75)

print(np.mean(list(test_carry.values())))
print(max(test_elo["ratings"].values()))
print(max(test_carry.values()))

1000.0
1131.8166644834728
1098.8624983626046


## 4. Elo Parameter Evaluation


In [9]:
def score_elo_params(all_games, seasons_tune, K, home_adv,
                     use_mov=True, mov_scale=1, use_home_adv=True,
                     scale=400, start_rating=1000, carry=0.6,
                     initial_ratings=None):

  ratings = {} if initial_ratings is None else initial_ratings.copy()
  ll_vec = []

  if not use_home_adv:
      home_adv = 0

  for s in seasons_tune:
      g_s = (all_games[all_games["season"] == s]
              .sort_values(["gameday", "game_id"])
              .reset_index(drop=True))

      ratings = carry_over(ratings, carry=carry)
      res = elo_season(games=g_s, ratings=ratings, K=K, scale=scale,
                       start_rating=start_rating, use_mov=use_mov,
                       mov_scale=mov_scale, use_home_adv=use_home_adv,
                       home_adv=home_adv)

      games_res = res["games"]
      non_ties = games_res["home_score"] != games_res["away_score"]

      p = games_res.loc[non_ties, "p_home"].to_numpy()
      y = (games_res.loc[non_ties, "home_score"] >
            games_res.loc[non_ties, "away_score"]).astype(float).to_numpy()

      ll_vec.extend(logloss(p, y))
      ratings = res["ratings"]

  return np.nanmean(ll_vec)

### Initial Parameter Test


In [10]:
test_score = score_elo_params(
    all_games=games,
    seasons_tune=[2021, 2022, 2023, 2024],
    K=20,
    home_adv=50,
    use_mov=True,
    mov_scale=calc_mov_scale(games[games["season"].isin([2021, 2022, 2023, 2024])]),
    use_home_adv=True,
    scale=400,
    start_rating=1000,
    carry=0.75
)

print(test_score)

0.6104861485408839


Compare `test_score` above with the unchanged naive 50/50 baseline:
$-\log(0.5)=0.6931$. The NFL notebook's recorded 0.6503 is not a college result.


## 5. Hyperparameter Tuning


In [11]:
from itertools import product

def tune_elo_params(all_games, seasons_tune, K_grid, home_adv_grid,
                    use_mov=True, mov_scale=1, use_home_adv=True,
                    scale=400, start_rating=1000, carry=0.6,
                    use_carry_grid=True,
                    carryover_grid=(0.35, 0.50, 0.65, 0.80),
                    initial_ratings=None):

    if not use_home_adv:
        home_adv_grid = [0]

    carry_grid = carryover_grid if use_carry_grid else [carry]
    results = []

    for K, home_adv, carry_val in product(K_grid, home_adv_grid, carry_grid):
        ll = score_elo_params(
            all_games=all_games, seasons_tune=seasons_tune,
            K=K, home_adv=home_adv,
            use_mov=use_mov, mov_scale=mov_scale,
            use_home_adv=use_home_adv, scale=scale,
            start_rating=start_rating, carry=carry_val,
            initial_ratings=initial_ratings
        )

        results.append({"K": K, "home_adv": home_adv, "carry": carry_val,
                        "logloss": ll})

    results = pd.DataFrame(results).sort_values("logloss").reset_index(drop=True)

    best = results.iloc[0]

    return {"K_best": best["K"], "home_adv_best": best["home_adv"],
            "carry_best": best["carry"], "logloss": best["logloss"],
            "results": results}

### Tuning Sanity Check


In [12]:
K_grid = [10, 15, 20, 25]
home_adv_grid = [25, 40, 55, 70]

mov_scale = calc_mov_scale(games[games["season"].isin([2021, 2022, 2023, 2024])])

test_tune = tune_elo_params(all_games=games, seasons_tune=[2021, 2022, 2023, 2024], K_grid=K_grid, 
                            home_adv_grid=home_adv_grid, mov_scale=mov_scale, carryover_grid=[0.50, 0.65, 0.75, 0.85])

print(test_tune["K_best"])
print(test_tune["home_adv_best"])
print(test_tune["carry_best"])
print(test_tune["logloss"])

test_tune["results"].head(10)

25.0
70.0
0.85
0.5973862752397552


,K,home_adv,carry,logloss
0,25,70,0.85,0.597386
1,25,70,0.75,0.599210
2,25,55,0.85,0.599768
3,25,70,0.65,0.601577
4,25,55,0.75,0.601740
5,25,40,0.85,0.603724
6,25,55,0.65,0.604247
7,20,70,0.85,0.604592
8,25,70,0.50,0.605851
9,25,40,0.75,0.605860


In [15]:
display(np.round(compress_exp_grid(15, 40, n=10, curve=-1)))
display(np.round(compress_exp_grid(15, 40, n=10, curve=1)))
display(np.round(compress_exp_grid(30, 75, n=10, curve=-1)))
display(np.round(compress_exp_grid(30, 75, n=10, curve=1)))

array([15., 19., 23., 26., 29., 32., 34., 36., 38., 40.])

array([15., 17., 19., 21., 23., 26., 29., 32., 36., 40.])

array([30., 37., 44., 50., 56., 60., 65., 68., 72., 75.])

array([30., 33., 37., 40., 45., 49., 55., 61., 68., 75.])

## 6. Prospective Expanding-Window Elo


In [16]:
def run_expanding_k_elo(all_games, first_train_year=2003, first_eval_year=2010,
                        K_grid=None, home_adv_grid=None, scale=400,
                        start_rating=1000, carry=0.6, use_carry_grid=True,
                        carryover_grid=(0.35, 0.50, 0.65, 0.80),
                        param_tune_window=None, use_mov=True, use_home_adv=True,
                        projection_season=None):

    if K_grid is None:
        K_grid = compress_exp_grid(1, 60, n=10, curve=-1)
    if home_adv_grid is None:
        home_adv_grid = compress_exp_grid(1, 60, n=10, curve=1)
    if not use_home_adv:
        home_adv_grid = [0]

    K_grid = np.atleast_1d(K_grid)
    home_adv_grid = np.atleast_1d(home_adv_grid)
    fixed_param_mode = len(K_grid) == 1 and len(home_adv_grid) == 1 and not use_carry_grid

    seasons = sorted(all_games["season"].unique())
    eval_years = [s for s in seasons if s >= first_eval_year]

    # Include the current schedule season even if it has no completed games yet.
    if projection_season is not None and projection_season not in eval_years:
        eval_years = sorted(set(eval_years + [projection_season]))

    all_out = []
    param_out = []
    ratings = {}
    ratings_before_season = {}

    for yr in eval_years:
        train_years = [s for s in seasons if first_train_year <= s < yr]
        if len(train_years) == 0:
            raise ValueError(f"No training seasons available before eval year: {yr}")

        param_tune_years = (
            train_years if param_tune_window is None
            else train_years[-param_tune_window:]
        )

        train_games = all_games[all_games["season"].isin(train_years)]
        mov_scale = calc_mov_scale(train_games) if use_mov else np.nan

        if fixed_param_mode:
            K_best = K_grid[0]
            home_adv_best = home_adv_grid[0] if use_home_adv else 0
            carry_best = carry
            best_logloss = np.nan
        else:
            param_fit = tune_elo_params(
                all_games=all_games, seasons_tune=param_tune_years,
                K_grid=K_grid, home_adv_grid=home_adv_grid,
                use_mov=use_mov, mov_scale=mov_scale,
                use_home_adv=use_home_adv, scale=scale,
                start_rating=start_rating, carry=carry,
                use_carry_grid=use_carry_grid,
                carryover_grid=carryover_grid,
                initial_ratings=ratings_before_season.get(param_tune_years[0], {})
            )

            K_best = param_fit["K_best"]
            home_adv_best = param_fit["home_adv_best"] if use_home_adv else 0
            carry_best = param_fit["carry_best"] if use_carry_grid else carry
            best_logloss = param_fit["logloss"]

        # Bootstrap training history once; later seasons keep the prior final ratings.
        if yr == eval_years[0]:
            for s in train_years:
                season_games = (all_games[all_games["season"] == s]
                                .sort_values(["gameday", "gametime", "game_id"])
                                .reset_index(drop=True))

                ratings_before_season[s] = ratings.copy()
                ratings = carry_over(ratings, carry=carry_best)
                res = elo_season(
                    games=season_games, ratings=ratings, K=K_best,
                    scale=scale, start_rating=start_rating,
                    use_mov=use_mov, mov_scale=mov_scale,
                    use_home_adv=use_home_adv, home_adv=home_adv_best
                )

                ratings = res["ratings"]

        ratings_before_season[yr] = ratings.copy()
        ratings = carry_over(ratings, carry=carry_best)

        eval_games = (all_games[all_games["season"] == yr]
                      .sort_values(["gameday", "gametime", "game_id"])
                      .reset_index(drop=True))

        res_eval = elo_season(
            games=eval_games, ratings=ratings, K=K_best,
            scale=scale, start_rating=start_rating,
            use_mov=use_mov, mov_scale=mov_scale,
            use_home_adv=use_home_adv, home_adv=home_adv_best
        )

        ratings = res_eval["ratings"]
        games_out = res_eval["games"].copy()
        games_out["K_used"] = K_best
        games_out["home_adv_used"] = np.where(games_out["location"] == "Neutral", 0, home_adv_best)
        games_out["carry_used"] = carry_best
        games_out["mov_scale_used"] = mov_scale
        games_out["use_mov"] = use_mov
        games_out["use_home_adv"] = use_home_adv
        games_out["train_start"] = min(train_years)
        games_out["train_end"] = max(train_years)
        games_out["param_tune_start"] = min(param_tune_years)
        games_out["param_tune_end"] = max(param_tune_years)
        games_out["param_tune_n"] = len(param_tune_years)
        games_out["param_tune_window"] = param_tune_window

        all_out.append(games_out)

        param_out.append({
            "eval_year": yr,
            "train_start": min(train_years),
            "train_end": max(train_years),
            "param_tune_start": min(param_tune_years),
            "param_tune_end": max(param_tune_years),
            "param_tune_n": len(param_tune_years),
            "param_tune_window": param_tune_window,
            "K_best": K_best,
            "home_adv_best": home_adv_best,
            "carry_best": carry_best,
            "mov_scale": mov_scale,
            "use_mov": use_mov,
            "use_home_adv": use_home_adv,
            "use_carry_grid": use_carry_grid,
            "logloss": best_logloss
        })

    return {
        "games": pd.concat(all_out, ignore_index=True),
        "K_by_year": pd.DataFrame(param_out),
        "ratings": res_eval["ratings"],
        "current_season": eval_years[-1]
    }

### Prospective Validation Test


In [18]:
K_grid=[15, 20, 25, 30, 35, 40, 45, 50] 
[x - 5 for x in K_grid]
K_grid = np.round(compress_exp_grid(15, 40, n=4, curve=1))
home_adv_grid = np.round(compress_exp_grid(35, 75, n=4, curve=1))
print(K_grid)
print(home_adv_grid)

[15. 21. 29. 40.]
[35. 44. 57. 75.]


In [19]:
# Exclude Preseason and Unplayed Games
elo_df = games[games["game_type"].isin(["REG", "POST"]) &
               games["home_score"].notna() &
               games["away_score"].notna()].copy()

test_expanding = run_expanding_k_elo(
    all_games=elo_df,
    first_train_year=2003,
    first_eval_year=2008,
    K_grid= K_grid, #[45, 55, 60],
    home_adv_grid= home_adv_grid, #[25, 35, 45],
    scale=400,
    start_rating=1000,
    use_carry_grid=True,
    carryover_grid=[0.35, 0.5, 0.65],
    param_tune_window=5,
    use_mov=True,
    use_home_adv=True
)

#Inspect
display(test_expanding["K_by_year"])
display(test_expanding["games"][["season", "gameday", "home_team", "away_team",
                         "elo_home_pre", "elo_away_pre", "p_home", "K_used",
                         "location", "home_adv_used", "carry_used", "mov_scale_used"]])

#Sanity Check
test_expanding["K_by_year"][["eval_year", "train_start", "train_end", "param_tune_start", "param_tune_end"]]

,eval_year,train_start,train_end,param_tune_start,param_tune_end,param_tune_n,param_tune_window,K_best,home_adv_best,carry_best,mov_scale,use_mov,use_home_adv,use_carry_grid,logloss
0,2008,2003,2007,2003,2007,5,5,40.0,75.0,0.65,2.621542,True,True,True,0.573104
1,2009,2003,2008,2004,2008,5,5,40.0,75.0,0.65,2.624574,True,True,True,0.565050
2,2010,2003,2009,2005,2009,5,5,40.0,75.0,0.65,2.617600,True,True,True,0.560608
3,2011,2003,2010,2006,2010,5,5,40.0,75.0,0.65,2.622160,True,True,True,0.555983
4,2012,2003,2011,2007,2011,5,5,40.0,75.0,0.65,2.623292,True,True,True,0.553818
5,2013,2003,2012,2008,2012,5,5,40.0,75.0,0.65,2.625893,True,True,True,0.549562
6,2014,2003,2013,2009,2013,5,5,40.0,75.0,0.65,2.636394,True,True,True,0.545305
7,2015,2003,2014,2010,2014,5,5,40.0,75.0,0.65,2.636293,True,True,True,0.549138
8,2016,2003,2015,2011,2015,5,5,40.0,75.0,0.65,2.639709,True,True,True,0.546817
9,2017,2003,2016,2012,2016,5,5,40.0,75.0,0.65,2.640334,True,True,True,0.552005


,season,gameday,home_team,away_team,elo_home_pre,elo_away_pre,p_home,K_used,location,home_adv_used,carry_used,mov_scale_used
0,2008,2008-08-28,2050,111,995.314259,980.918079,0.625886,40.0,Home,75.0,0.65,2.621542
1,2008,2008-08-28,2084,2638,930.963945,936.695633,0.598385,40.0,Home,75.0,0.65,2.621542
2,2008,2008-08-28,2117,2197,1032.200600,964.035008,0.695112,40.0,Home,75.0,0.65,2.621542
3,2008,2008-08-28,2199,282,918.489391,960.989471,0.546635,40.0,Home,75.0,0.65,2.621542
4,2008,2008-08-28,41,2275,1039.672074,988.758276,0.673667,40.0,Home,75.0,0.65,2.621542
...,...,...,...,...,...,...,...,...,...,...,...,...
15744,2026,2026-10-15,295,290,1027.355916,999.082763,0.644394,40.0,Home,75.0,0.65,2.638913
15745,2026,2026-10-15,326,36,1053.599210,910.846927,0.777902,40.0,Home,75.0,0.65,2.638913
15746,2026,2026-10-16,2655,235,1067.874058,1104.567308,0.554906,40.0,Home,75.0,0.65,2.638913
15747,2026,2026-10-16,2509,264,970.909977,1074.866424,0.458425,40.0,Home,75.0,0.65,2.638913


,eval_year,train_start,train_end,param_tune_start,param_tune_end
0,2008,2003,2007,2003,2007
1,2009,2003,2008,2004,2008
2,2010,2003,2009,2005,2009
3,2011,2003,2010,2006,2010
4,2012,2003,2011,2007,2011
5,2013,2003,2012,2008,2012
6,2014,2003,2013,2009,2013
7,2015,2003,2014,2010,2014
8,2016,2003,2015,2011,2015
9,2017,2003,2016,2012,2016


In [20]:
display(test_expanding["K_by_year"])
display(test_expanding["games"][["season", "gameday", "home_team", "away_team", "elo_home_pre", "elo_away_pre", "p_home", "K_used", "location", "home_adv_used", "carry_used", "mov_scale_used"]])

,eval_year,train_start,train_end,param_tune_start,param_tune_end,param_tune_n,param_tune_window,K_best,home_adv_best,carry_best,mov_scale,use_mov,use_home_adv,use_carry_grid,logloss
0,2008,2003,2007,2003,2007,5,5,40.0,75.0,0.65,2.621542,True,True,True,0.573104
1,2009,2003,2008,2004,2008,5,5,40.0,75.0,0.65,2.624574,True,True,True,0.565050
2,2010,2003,2009,2005,2009,5,5,40.0,75.0,0.65,2.617600,True,True,True,0.560608
3,2011,2003,2010,2006,2010,5,5,40.0,75.0,0.65,2.622160,True,True,True,0.555983
4,2012,2003,2011,2007,2011,5,5,40.0,75.0,0.65,2.623292,True,True,True,0.553818
5,2013,2003,2012,2008,2012,5,5,40.0,75.0,0.65,2.625893,True,True,True,0.549562
6,2014,2003,2013,2009,2013,5,5,40.0,75.0,0.65,2.636394,True,True,True,0.545305
7,2015,2003,2014,2010,2014,5,5,40.0,75.0,0.65,2.636293,True,True,True,0.549138
8,2016,2003,2015,2011,2015,5,5,40.0,75.0,0.65,2.639709,True,True,True,0.546817
9,2017,2003,2016,2012,2016,5,5,40.0,75.0,0.65,2.640334,True,True,True,0.552005


,season,gameday,home_team,away_team,elo_home_pre,elo_away_pre,p_home,K_used,location,home_adv_used,carry_used,mov_scale_used
0,2008,2008-08-28,2050,111,995.314259,980.918079,0.625886,40.0,Home,75.0,0.65,2.621542
1,2008,2008-08-28,2084,2638,930.963945,936.695633,0.598385,40.0,Home,75.0,0.65,2.621542
2,2008,2008-08-28,2117,2197,1032.200600,964.035008,0.695112,40.0,Home,75.0,0.65,2.621542
3,2008,2008-08-28,2199,282,918.489391,960.989471,0.546635,40.0,Home,75.0,0.65,2.621542
4,2008,2008-08-28,41,2275,1039.672074,988.758276,0.673667,40.0,Home,75.0,0.65,2.621542
...,...,...,...,...,...,...,...,...,...,...,...,...
15744,2026,2026-10-15,295,290,1027.355916,999.082763,0.644394,40.0,Home,75.0,0.65,2.638913
15745,2026,2026-10-15,326,36,1053.599210,910.846927,0.777902,40.0,Home,75.0,0.65,2.638913
15746,2026,2026-10-16,2655,235,1067.874058,1104.567308,0.554906,40.0,Home,75.0,0.65,2.638913
15747,2026,2026-10-16,2509,264,970.909977,1074.866424,0.458425,40.0,Home,75.0,0.65,2.638913


### Out-of-Sample Performance


In [21]:
#Calculate out-of-sample performance for all evaluation seasons
eval_games = test_expanding["games"].copy()
non_ties = eval_games["home_score"] != eval_games["away_score"]

eval_games["actual_home_win"] = (
    eval_games["home_score"] > eval_games["away_score"]).astype(int)

eval_games["pred_home_win"] = (eval_games["p_home"] >= 0.5).astype(int)

eval_games["logloss"] = np.nan
eval_games.loc[non_ties, "logloss"] = logloss(
    eval_games.loc[non_ties, "p_home"],
    eval_games.loc[non_ties, "actual_home_win"]
)

performance_by_year = (eval_games.loc[non_ties].groupby("season").agg(
    games=("game_id", "size"),
    accuracy=("pred_home_win",
              lambda x: (x == eval_games.loc[x.index, "actual_home_win"]).mean()),
    logloss=("logloss", "mean")).reset_index())

print(performance_by_year)

    season  games  accuracy   logloss
0     2008    804  0.732587  0.553538
1     2009    808  0.740099  0.539607
2     2010    808  0.724010  0.560921
3     2011    812  0.740148  0.547195
4     2012    840  0.719048  0.546427
5     2013    848  0.734670  0.531983
6     2014    868  0.732719  0.558511
7     2015    870  0.736782  0.548781
8     2016    873  0.701031  0.572875
9     2017    874  0.712815  0.567379
10    2018    884  0.728507  0.546478
11    2019    888  0.748874  0.538221
12    2020    568  0.697183  0.575797
13    2021    887  0.720406  0.564735
14    2022    896  0.700893  0.583380
15    2023    910  0.712088  0.562464
16    2024    919  0.700762  0.573847
17    2025    934  0.739829  0.553857
18    2026    458  0.759825  0.538958


In [22]:
overall_accuracy = (eval_games.loc[non_ties, "pred_home_win"] ==
                    eval_games.loc[non_ties, "actual_home_win"]).mean()

overall_logloss = eval_games.loc[non_ties, "logloss"].mean()

print("Accuracy:", overall_accuracy)
print("Log Loss:", overall_logloss)

Accuracy: 0.7248079243126547
Log Loss: 0.5563535381356333


In [23]:
accuracy_by_year = (
    eval_games.loc[non_ties]
    .groupby("season")
    .apply(lambda x: (x["pred_home_win"] == x["actual_home_win"]).mean())
    .reset_index(name="accuracy")
)

accuracy_by_year

,season,accuracy
0,2008,0.732587
1,2009,0.740099
2,2010,0.724010
3,2011,0.740148
4,2012,0.719048
5,2013,0.734670
6,2014,0.732719
7,2015,0.736782
8,2016,0.701031
9,2017,0.712815


## 7. Final Prospective Elo Model


In [31]:
K_grid = np.round(compress_exp_grid(15, 40, n=7, curve=1))
home_adv_grid = np.round(compress_exp_grid(35, 75, n=7, curve=1))
carry_grid = np.round(compress_exp_grid(45, 80, n=7, curve=1))*.01

display(K_grid)
display(home_adv_grid)
display(carry_grid)

array([15., 18., 21., 24., 29., 34., 40.])

array([35., 39., 44., 50., 57., 65., 75.])

array([0.45, 0.49, 0.53, 0.58, 0.64, 0.72, 0.8 ])

In [32]:
K_grid = np.round(compress_exp_grid(15, 40, n=7, curve=1))
home_adv_grid = np.round(compress_exp_grid(35, 75, n=7, curve=1))
carry_grid = np.round(compress_exp_grid(45, 80, n=7, curve=1))*.01
cfb_elo = run_expanding_k_elo(all_games=elo_df, first_train_year=2003,
                    first_eval_year=2008,
                    K_grid = K_grid, # compress_exp_grid(15, 60, n=10, curve=-1),
                    home_adv_grid = home_adv_grid, # [25, 30, 35, 40, 50, 53, 55, 58, 60], # compress_exp_grid(15, 60, n=10, curve=1),
                    scale=400, start_rating=1000, use_carry_grid=True,
                    carryover_grid=carry_grid,
                    param_tune_window=5, use_mov=True, use_home_adv=True)


### Performance Check


In [33]:
eval_games = cfb_elo["games"].query("home_score != away_score").copy()

eval_games["actual"] = (eval_games["home_score"] > eval_games["away_score"]).astype(int)
eval_games["pred"] = (eval_games["p_home"] >= 0.5).astype(int)
eval_games["correct"] = eval_games["pred"] == eval_games["actual"]
eval_games["ll"] = -(eval_games["actual"] * np.log(eval_games["p_home"]) + (1 - eval_games["actual"]) * np.log1p(-eval_games["p_home"]))

print("Accuracy:", accuracy_score(eval_games["actual"], eval_games["pred"]))
print("Log Loss:", log_loss(eval_games["actual"], eval_games["p_home"]))

Accuracy: 0.7267128071623595
Log Loss: 0.5437364406122526


In [34]:
performance_by_year = eval_games.groupby("season").agg(
    games=("game_id", "size"), accuracy=("correct", "mean"),
    logloss=("ll", "mean")).reset_index()

performance_by_year

,season,games,accuracy,logloss
0,2008,804,0.728856,0.542231
1,2009,808,0.748762,0.524307
2,2010,808,0.731436,0.550672
3,2011,812,0.741379,0.534665
4,2012,840,0.721429,0.533732
5,2013,848,0.734670,0.519164
6,2014,868,0.728111,0.542077
7,2015,870,0.741379,0.533346
8,2016,873,0.699885,0.566644
9,2017,874,0.710526,0.555585


### Tuned Parameter Selection


In [35]:
params = cfb_elo["K_by_year"].copy()
params[["eval_year", "K_best", "home_adv_best", "carry_best", "mov_scale", "logloss"]]

,eval_year,K_best,home_adv_best,carry_best,mov_scale,logloss
0,2008,40.0,75.0,0.8,2.621542,0.567268
1,2009,40.0,75.0,0.8,2.624574,0.556883
2,2010,40.0,75.0,0.8,2.617600,0.550604
3,2011,40.0,75.0,0.8,2.622160,0.545189
4,2012,40.0,75.0,0.8,2.623292,0.542197
5,2013,40.0,75.0,0.8,2.625893,0.537134
6,2014,40.0,75.0,0.8,2.636394,0.532560
7,2015,40.0,75.0,0.8,2.636293,0.536122
8,2016,40.0,65.0,0.8,2.639709,0.532682
9,2017,40.0,65.0,0.8,2.640334,0.538848


In [ ]:
print("K values:")
print(params["K_best"].value_counts().sort_index())

print("\nHome advantage values:")
print(params["home_adv_best"].value_counts().sort_index())

print("\nCarry values:")
print(params["carry_best"].value_counts().sort_index())
print(params[["K_best", "home_adv_best", "carry_best"]].describe())

display(np.median(params["K_best"]))

K values:
K_best
40.0    19
Name: count, dtype: int64

Home advantage values:
home_adv_best
65.0     4
75.0    15
Name: count, dtype: int64

Carry values:
carry_best
0.8    19
Name: count, dtype: int64


,K_best,home_adv_best,carry_best
count,19.0,19.000000,1.900000e+01
mean,40.0,72.894737,8.000000e-01
std,0.0,4.188539,1.140646e-16
min,40.0,65.000000,8.000000e-01
25%,40.0,75.000000,8.000000e-01
50%,40.0,75.000000,8.000000e-01
75%,40.0,75.000000,8.000000e-01
max,40.0,75.000000,8.000000e-01


### Rerun Elo Tuning (Tightened Grids)


In [ ]:
K_grid = [47, 50, 53, 56]#np.round(compress_exp_grid(40, 55, n=4, curve=-1))
home_adv_grid = [65, 70, 75, 80, 85]#np.round(compress_exp_grid(65, 80, n=3, curve=1))
carry_grid = [0.85, 0.9, 0.95]
print(K_grid)
print(home_adv_grid)
print(carry_grid)

[40. 47. 52. 55.]
[65. 71. 80.]
[0.8, 0.85, 0.9]


In [65]:
projection_season = int(games.loc[games["game_type"].isin(["REG", "POST"]),"season"].max())

K_grid = [47, 50, 53, 56]#np.round(compress_exp_grid(40, 55, n=4, curve=-1))
home_adv_grid = [67, 70, 73, 76]#np.round(compress_exp_grid(65, 80, n=3, curve=1))
carry_grid = [0.85, 0.9, 0.95]

cfb_elo = run_expanding_k_elo(
    all_games=elo_df,
    first_train_year=2005,
    first_eval_year=2010,
    K_grid=K_grid,
    home_adv_grid=home_adv_grid, 
    scale=400,
    start_rating=1000,
    use_carry_grid=True,
    carryover_grid=carry_grid,
    param_tune_window=5,
    use_mov=True,
    use_home_adv=True,
    projection_season=projection_season
)


In [68]:
eval_games = cfb_elo["games"].query("home_score != away_score").copy()
eval_games["actual"] = (eval_games["home_score"] > eval_games["away_score"]).astype(int)
eval_games["pred"] = (eval_games["p_home"] >= 0.5).astype(int)
eval_games["correct"] = eval_games["pred"] == eval_games["actual"]
eval_games["ll"] = -(eval_games["actual"] * np.log(eval_games["p_home"]) + (1 - eval_games["actual"]) * np.log1p(-eval_games["p_home"]))

print("Accuracy:", accuracy_score(eval_games["actual"], eval_games["pred"]))
print("Log Loss:", log_loss(eval_games["actual"], eval_games["p_home"]))

performance_by_year = eval_games.groupby("season").agg(
    games=("game_id", "size"), accuracy=("correct", "mean"),
    logloss=("ll", "mean")).reset_index()

display(performance_by_year)

params = cfb_elo["K_by_year"].copy()
params[["eval_year", "K_best", "home_adv_best", "carry_best", "mov_scale", "logloss"]].rename(columns = {"logloss":"tuning_logloss"})

Accuracy: 0.7324750654311382
Log Loss: 0.523675563335924


,season,games,accuracy,logloss
0,2010,808,0.738861,0.537086
1,2011,812,0.745074,0.519599
2,2012,840,0.741667,0.512147
3,2013,848,0.737028,0.495844
4,2014,868,0.744240,0.521228
5,2015,870,0.758621,0.505786
6,2016,873,0.702176,0.557901
7,2017,874,0.712815,0.535695
8,2018,884,0.743213,0.504582
9,2019,888,0.769144,0.495640


,eval_year,K_best,home_adv_best,carry_best,mov_scale,tuning_logloss
0,2010,56.0,73.0,0.95,2.599780,0.550054
1,2011,56.0,73.0,0.95,2.608829,0.535432
2,2012,56.0,76.0,0.95,2.612222,0.530020
3,2013,56.0,73.0,0.95,2.616890,0.521378
4,2014,56.0,70.0,0.95,2.630614,0.513949
5,2015,56.0,67.0,0.95,2.631115,0.516880
6,2016,56.0,67.0,0.95,2.635602,0.510774
7,2017,56.0,67.0,0.95,2.636686,0.518531
8,2018,56.0,67.0,0.90,2.638112,0.523308
9,2019,56.0,67.0,0.95,2.641580,0.524899


## 8. Elo Win Probability to Spread Map

Validate the Elo-to-spread mapping prospectively with an expanding window.

For each evaluation season, the spread coefficient is fit only on completed games from earlier seasons, then applied to that season's already-prospective Elo win probabilities. This produces a leakage-safe historical spread validation.

After validation, a final production coefficient is fit on all completed historical games and used only for future projections.


In [69]:
from scipy.optimize import minimize_scalar

spread_games = cfb_elo["games"].dropna(subset=["home_score", "away_score", "p_home"]).copy()

spread_games = spread_games.sort_values(["season", "gameday", "gametime", "game_id"]).reset_index(drop=True)

spread_games["home_margin"] = (spread_games["home_score"] - spread_games["away_score"])

p = spread_games["p_home"].clip(1e-6, 1 - 1e-6)
spread_games["logit_p_home"] = np.log(p / (1 - p))

spread_seasons = sorted(spread_games["season"].unique())
spread_validation = []

for eval_season in spread_seasons[1:]:
    train = spread_games[spread_games["season"] < eval_season].copy()
    test = spread_games[spread_games["season"] == eval_season].copy()

    spread_opt = minimize_scalar(
        lambda slope: np.mean(
            np.abs(
                train["home_margin"]
                - slope * train["logit_p_home"]
            )
        ),
        bounds=(0, 20),
        method="bounded"
    )

    slope = spread_opt.x

    test["spread_slope_used"] = slope
    test["elo_home_margin"] = slope * test["logit_p_home"]
    test["model_home_spread"] = -(np.round(test["elo_home_margin"] * 2) / 2)
    test["spread_abs_error"] = (test["home_margin"] - test["elo_home_margin"]).abs()

    spread_validation.append(test)

spread_validation = pd.concat(spread_validation, ignore_index=True)

print("Prospective spread MAE:", spread_validation["spread_abs_error"].mean())
print("Fraction actual margin above predicted margin:", (spread_validation["home_margin"] > spread_validation["elo_home_margin"]).mean())


Prospective spread MAE: 14.321880929190431
Fraction actual margin above predicted margin: 0.5164678520519169


In [70]:
spread_performance_by_year = (spread_validation.groupby("season").agg(games=("game_id", "size"), spread_slope_used=("spread_slope_used", "first"),
                                                                      spread_mae=("spread_abs_error", "mean")).reset_index())

display(spread_performance_by_year)

,season,games,spread_slope_used,spread_mae
0,2011,812,11.045630,13.715891
1,2012,840,10.886609,14.231456
2,2013,848,10.796046,14.473824
3,2014,868,10.886603,14.615388
4,2015,870,10.389581,14.802877
5,2016,873,10.134457,14.922686
6,2017,874,10.014959,14.490177
7,2018,884,9.947678,14.455620
8,2019,888,10.041510,13.637199
9,2020,568,10.134458,14.056258


### Production Spread Coefficient

Fit one final coefficient on all completed historical games. This coefficient is not used to score the historical validation above; it is used only for future projections.


In [71]:
production_spread_opt = minimize_scalar(
    lambda slope: np.mean(
        np.abs(
            spread_games["home_margin"]
            - slope * spread_games["logit_p_home"]
        )
    ),
    bounds=(0, 20),
    method="bounded"
)

spread_slope = production_spread_opt.x

print(f"Production spread slope: {spread_slope:.4f}")


Production spread slope: 10.3107


## 9. Current Elo Ratings

Use the current ratings returned directly by the final expanding Elo fit, following the same
2003-forward training/replay path as the fitted model. Keep individual non-FBS opponent ratings
for projections; publish rankings only for teams classified as FBS in the current season.


In [74]:
current_season = cfb_elo["current_season"]
params = cfb_elo["K_by_year"].set_index("eval_year").loc[current_season]

# Use the final expanding fit's current Elo state directly.
ratings = cfb_elo["ratings"].copy()

current_teams = pd.unique(
    games.loc[
        games["season"] == current_season,
        ["home_team", "away_team"]
    ].values.ravel()
)

# As in the original model, a newly appearing team starts at 1000.
ratings = {team: ratings.get(team, 1000) for team in current_teams}

current_fbs_teams = pd.unique(np.concatenate([
    games.loc[(games["season"] == current_season) & (games["home_division"] == "fbs"), "home_team"],
    games.loc[(games["season"] == current_season) & (games["away_division"] == "fbs"), "away_team"]
]))
team_names = pd.concat([
    games[["home_team", "home_team_name"]].rename(columns={"home_team": "team", "home_team_name": "team_name"}),
    games[["away_team", "away_team_name"]].rename(columns={"away_team": "team", "away_team_name": "team_name"})
]).drop_duplicates("team", keep="last").set_index("team")["team_name"]

current_ratings = (
    pd.Series(ratings, name="elo")
    .rename_axis("team")
    .reset_index()
    .query("team in @current_fbs_teams")
    .sort_values("elo", ascending=False)
    .reset_index(drop=True)
)
current_ratings["team_name"] = current_ratings["team"].map(team_names)
current_ratings.insert(0, "rank", range(1, len(current_ratings) + 1))

current_ratings


,rank,team,elo,team_name
0,1,61,1549.485329,Georgia
1,2,194,1541.605839,Ohio State
2,3,84,1528.583010,Indiana
3,4,87,1496.818614,Notre Dame
4,5,2390,1452.866093,Miami
...,...,...,...,...
133,134,2534,808.991352,Sam Houston
134,135,2050,808.619097,Ball State
135,136,2433,755.199860,UL Monroe
136,137,2638,745.712009,UTEP


## 10. Daily Rankings and Projections

Use current Elo ratings to identify games scheduled for a selected date and calculate each team's pregame Elo and win probability.


In [75]:
as_of_date = pd.Timestamp.now(tz="America/Chicago").date()
rank_map = current_ratings.set_index("team")["rank"]

future_projections = games[
    (games["season"] == current_season) &
    games["game_type"].isin(["REG", "POST"]) &
    games["home_score"].isna() &
    games["away_score"].isna() &
    (games["status"].isna() | games["status"].eq("STATUS_SCHEDULED")) &
    (pd.to_datetime(games["gameday"]).dt.date >= as_of_date)
].copy()

future_projections["home_elo"] = future_projections["home_team"].map(ratings).fillna(1000)
future_projections["away_elo"] = future_projections["away_team"].map(ratings).fillna(1000)
future_projections["home_rank"] = future_projections["home_team"].map(rank_map)
future_projections["away_rank"] = future_projections["away_team"].map(rank_map)

future_projections["game_home_adv"] = np.where(future_projections["location"] == "Neutral", 0, params["home_adv_best"])
future_projections["p_home"] = 1 / (1 + 10 ** ((future_projections["away_elo"] - future_projections["home_elo"] - future_projections["game_home_adv"]) / 400))
future_projections["p_away"] = 1 - future_projections["p_home"]

projection_p = future_projections["p_home"].clip(1e-6, 1 - 1e-6)
future_projections["elo_home_margin"] = spread_slope * np.log(projection_p / (1 - projection_p))
future_projections["model_home_spread"] = -(np.round(future_projections["elo_home_margin"] * 2) / 2)
future_projections["as_of_date"] = as_of_date

future_projections = future_projections[["as_of_date", "game_type", "week", "gameday", "gametime", "away_team", "home_team", "away_team_name", "home_team_name", "away_rank", "home_rank", "away_elo", "home_elo", "p_away", "p_home", "elo_home_margin", "model_home_spread"]
                                        ].sort_values(["gameday", "gametime"]).reset_index(drop=True)

daily_projections = future_projections[pd.to_datetime(future_projections["gameday"]).dt.date == as_of_date].reset_index(drop=True)

# College postseason week numbers restart, so match both phase and week.
current_week = future_projections.iloc[0]["week"] if not future_projections.empty else None
current_game_type = future_projections.iloc[0]["game_type"] if not future_projections.empty else None
weekly_projections = future_projections[
    (future_projections["week"] == current_week) &
    (future_projections["game_type"] == current_game_type)
].reset_index(drop=True)

daily_rankings = current_ratings.assign(as_of_date=as_of_date)

display(future_projections)    # every remaining game using current Elo
display(daily_projections)     # today's games
display(weekly_projections)    # current/upcoming NCAA FBS week
display(daily_rankings)        # current league-wide rankings


,as_of_date,game_type,week,gameday,gametime,away_team,home_team,away_team_name,home_team_name,away_rank,home_rank,away_elo,home_elo,p_away,p_home,elo_home_margin,model_home_spread
0,2026-10-17,REG,7,2026-10-17,11:00:00,344,99,Mississippi State,LSU,58.0,21,1139.019235,1295.521234,0.216432,0.783568,13.265521,-13.5
1,2026-10-17,REG,7,2026-10-17,11:00:00,57,251,Florida,Texas,45.0,7,1172.009976,1431.548497,0.132423,0.867577,19.381051,-19.5
2,2026-10-17,REG,7,2026-10-17,11:00:00,2132,277,Cincinnati,West Virginia,64.0,87,1131.853953,1036.540733,0.540656,0.459344,-1.680475,1.5
3,2026-10-17,REG,7,2026-10-17,11:00:00,2305,2306,Kansas,Kansas State,75.0,35,1077.857315,1194.434675,0.257930,0.742070,10.895873,-11.0
4,2026-10-17,REG,7,2026-10-17,11:00:00,2127,228,Charleston Southern,Clemson,NaN,30,844.009505,1235.432166,0.066677,0.933323,27.208775,-27.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
425,2026-10-17,REG,13,2026-11-28,14:30:00,55,48,Jacksonville State,Delaware,71.0,116,1086.893973,943.739539,0.607873,0.392127,-4.519997,4.5
426,2026-10-17,REG,13,2026-11-28,15:30:00,2005,167,Air Force,New Mexico,85.0,60,1041.422237,1133.739183,0.285549,0.714451,9.455944,-9.5
427,2026-10-17,REG,13,2026-11-28,20:00:00,2440,2439,Nevada,UNLV,123.0,42,892.207720,1176.428289,0.116934,0.883066,20.846006,-21.0
428,2026-10-17,REG,13,2026-11-28,22:00:00,16,62,Sacramento State,Hawai'i,133.0,112,819.175169,955.093959,0.237200,0.762800,12.043845,-12.0


,as_of_date,game_type,week,gameday,gametime,away_team,home_team,away_team_name,home_team_name,away_rank,home_rank,away_elo,home_elo,p_away,p_home,elo_home_margin,model_home_spread
0,2026-10-17,REG,7,2026-10-17,11:00:00,344,99,Mississippi State,LSU,58.0,21,1139.019235,1295.521234,0.216432,0.783568,13.265521,-13.5
1,2026-10-17,REG,7,2026-10-17,11:00:00,57,251,Florida,Texas,45.0,7,1172.009976,1431.548497,0.132423,0.867577,19.381051,-19.5
2,2026-10-17,REG,7,2026-10-17,11:00:00,2132,277,Cincinnati,West Virginia,64.0,87,1131.853953,1036.540733,0.540656,0.459344,-1.680475,1.5
3,2026-10-17,REG,7,2026-10-17,11:00:00,2305,2306,Kansas,Kansas State,75.0,35,1077.857315,1194.434675,0.257930,0.742070,10.895873,-11.0
4,2026-10-17,REG,7,2026-10-17,11:00:00,2127,228,Charleston Southern,Clemson,NaN,30,844.009505,1235.432166,0.066677,0.933323,27.208775,-27.0
5,2026-10-17,REG,7,2026-10-17,11:00:00,221,103,Pittsburgh,Boston College,39.0,104,1185.779345,981.590742,0.687771,0.312229,-8.142559,8.0
6,2026-10-17,REG,7,2026-10-17,11:00:00,258,2567,Virginia,SMU,55.0,18,1150.900664,1307.215121,0.216615,0.783385,13.254390,-13.5
7,2026-10-17,REG,7,2026-10-17,11:00:00,194,84,Ohio State,Indiana,2.0,3,1541.605839,1528.583010,0.422940,0.577060,3.203709,-3.0
8,2026-10-17,REG,7,2026-10-17,11:00:00,77,127,Northwestern,Michigan State,37.0,97,1189.619239,1002.166443,0.666719,0.333281,-7.149239,7.0
9,2026-10-17,REG,7,2026-10-17,11:00:00,2226,349,Florida Atlantic,Army,105.0,48,981.102152,1161.739501,0.193798,0.806202,14.698027,-14.5


,as_of_date,game_type,week,gameday,gametime,away_team,home_team,away_team_name,home_team_name,away_rank,home_rank,away_elo,home_elo,p_away,p_home,elo_home_margin,model_home_spread
0,2026-10-17,REG,7,2026-10-17,11:00:00,344,99,Mississippi State,LSU,58.0,21,1139.019235,1295.521234,0.216432,0.783568,13.265521,-13.5
1,2026-10-17,REG,7,2026-10-17,11:00:00,57,251,Florida,Texas,45.0,7,1172.009976,1431.548497,0.132423,0.867577,19.381051,-19.5
2,2026-10-17,REG,7,2026-10-17,11:00:00,2132,277,Cincinnati,West Virginia,64.0,87,1131.853953,1036.540733,0.540656,0.459344,-1.680475,1.5
3,2026-10-17,REG,7,2026-10-17,11:00:00,2305,2306,Kansas,Kansas State,75.0,35,1077.857315,1194.434675,0.257930,0.742070,10.895873,-11.0
4,2026-10-17,REG,7,2026-10-17,11:00:00,2127,228,Charleston Southern,Clemson,NaN,30,844.009505,1235.432166,0.066677,0.933323,27.208775,-27.0
5,2026-10-17,REG,7,2026-10-17,11:00:00,221,103,Pittsburgh,Boston College,39.0,104,1185.779345,981.590742,0.687771,0.312229,-8.142559,8.0
6,2026-10-17,REG,7,2026-10-17,11:00:00,258,2567,Virginia,SMU,55.0,18,1150.900664,1307.215121,0.216615,0.783385,13.254390,-13.5
7,2026-10-17,REG,7,2026-10-17,11:00:00,194,84,Ohio State,Indiana,2.0,3,1541.605839,1528.583010,0.422940,0.577060,3.203709,-3.0
8,2026-10-17,REG,7,2026-10-17,11:00:00,77,127,Northwestern,Michigan State,37.0,97,1189.619239,1002.166443,0.666719,0.333281,-7.149239,7.0
9,2026-10-17,REG,7,2026-10-17,11:00:00,2226,349,Florida Atlantic,Army,105.0,48,981.102152,1161.739501,0.193798,0.806202,14.698027,-14.5


,rank,team,elo,team_name,as_of_date
0,1,61,1549.485329,Georgia,2026-10-17
1,2,194,1541.605839,Ohio State,2026-10-17
2,3,84,1528.583010,Indiana,2026-10-17
3,4,87,1496.818614,Notre Dame,2026-10-17
4,5,2390,1452.866093,Miami,2026-10-17
...,...,...,...,...,...
133,134,2534,808.991352,Sam Houston,2026-10-17
134,135,2050,808.619097,Ball State,2026-10-17
135,136,2433,755.199860,UL Monroe,2026-10-17
136,137,2638,745.712009,UTEP,2026-10-17


In [76]:
### Save Data
current_ratings.to_csv("current_elo_rankings.csv", index=False)
daily_projections.to_csv("daily_projections.csv", index=False)
weekly_projections.to_csv("weekly_projections.csv", index=False)
future_projections.to_csv("future_projections.csv", index=False)
# historical_elo.to_csv("cfb_elo_history.csv", index=False)

### Adaptation and verification notes

Cell count, section order, modeling functions, initial 1000, Elo scale 400, MOV adjustment,
offseason regression, tuning grids, expanding windows, binary outcome evaluation, and spread
fit are preserved. The only function edit is renaming `nfl_home_result` to `cfb_home_result`
(and its error message). Data loading, phase filters, FBS ranking membership, and name labels
are college adaptations. OT weighting is inherited by request; it is not an NCAA requirement.

Two original execution defects are repaired here: `home_adv_grid=display(...)` lacked a comma
and returned `None`, and selecting an upcoming week failed when no games remained. A stale
NFL score and stale replay-year description are replaced with college-appropriate explanations.

See `VALIDATION.md` for the exact tests and their limits. Saved outputs from the NFL notebook
were cleared; they must not be interpreted as NCAA results. The delivered full tuning grids
are unchanged; reduced-grid verification runs do not establish their optimality on college data.
